In [19]:
import pandas as pd
import numpy as np
import re
import string

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize

# ML
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

# Evaluation
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [20]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [21]:
df = pd.read_csv("IMDB Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [22]:
print(df.shape)
print(df['sentiment'].value_counts())

df.sample(5)

(50000, 2)
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


,review,sentiment
34928,Is it just me or is that kid really annoying?<...,negative
43082,80's sleazy (glam)rock is like 90's house musi...,negative
42762,The only redeeming feature of this movie is St...,negative
21818,"This movie ""Joshua"" is extremely disturbing, a...",negative
34415,I pretty much liked every character on this sh...,negative


In [23]:
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    # Lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)

    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # Tokenization
    tokens = word_tokenize(text)

    # Remove stopwords
    tokens = [word for word in tokens if word not in stop_words]

    # Lemmatization
    tokens = [lemmatizer.lemmatize(word) for word in tokens]

    return " ".join(tokens)

In [24]:
df['cleaned_text'] = df['review'].apply(preprocess_text)

In [25]:
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})

In [26]:
X_train, X_test, y_train, y_test = train_test_split(
    df['cleaned_text'], df['sentiment'], test_size=0.2, random_state=42
)

In [27]:
bow = CountVectorizer(max_features=5000)

X_train_bow = bow.fit_transform(X_train)
X_test_bow = bow.transform(X_test)

In [28]:
tfidf = TfidfVectorizer(max_features=5000)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [29]:
lr = LogisticRegression()
lr.fit(X_train_tfidf, y_train)

y_pred_lr = lr.predict(X_test_tfidf)

In [30]:
dt = DecisionTreeClassifier()
dt.fit(X_train_tfidf, y_train)

y_pred_dt = dt.predict(X_test_tfidf)

In [31]:
def evaluate_model(y_true, y_pred, model_name):
    print(f"\n{model_name}")
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("Precision:", precision_score(y_true, y_pred))
    print("Recall:", recall_score(y_true, y_pred))
    print("F1 Score:", f1_score(y_true, y_pred))

In [33]:
nb = MultinomialNB()
nb.fit(X_train_tfidf, y_train)

y_pred_nb = nb.predict(X_test_tfidf)

In [34]:
evaluate_model(y_test, y_pred_lr, "Logistic Regression")
evaluate_model(y_test, y_pred_nb, "Naive Bayes")
evaluate_model(y_test, y_pred_dt, "Decision Tree")


Logistic Regression
Accuracy: 0.8875
Precision: 0.8777992277992278
Recall: 0.9023615796785076
F1 Score: 0.889910950190821

Naive Bayes
Accuracy: 0.8543
Precision: 0.8529759558533702
Recall: 0.8589005755110141
F1 Score: 0.8559280134480372

Decision Tree
Accuracy: 0.7149
Precision: 0.7195024077046549
Recall: 0.7116491367334788
F1 Score: 0.7155542252818518


# Final Verdicts & Insights
Logistic Regression emerged as the best-performing model with the highest accuracy and F1 score, making it the most suitable choice for this sentiment analysis task. Its ability to handle high-dimensional sparse data like TF-IDF features contributed to its strong performance.
Naive Bayes showed good performance with relatively high accuracy and faster training time. However, its assumption of feature independence slightly limited its effectiveness compared to Logistic Regression.
Decision Tree performed the weakest among all models. This is likely due to overfitting and its inability to effectively handle sparse text-based features, resulting in lower accuracy and F1 score.
TF-IDF proved to be an effective feature engineering technique as it assigns importance to meaningful words while reducing the impact of frequently occurring but less informative terms.
Text preprocessing steps such as lowercasing, stopword removal, tokenization, and lemmatization played a crucial role in improving model performance by cleaning and standardizing the input data.
Overall, the combination of proper preprocessing, TF-IDF vectorization, and Logistic Regression resulted in the most reliable and accurate sentiment classification.

# Conclusion

This project demonstrated the complete implementation of a sentiment analysis system using an NLP pipeline and machine learning models. Starting from raw text data, various preprocessing steps such as lowercasing, punctuation removal, stopword removal, tokenization, and lemmatization were applied to clean and standardize the data. These steps played a crucial role in improving the overall quality of the input.

Feature engineering techniques like Bag of Words and TF-IDF were used to convert textual data into numerical form. Among them, TF-IDF provided better results as it effectively captured the importance of words while reducing the impact of commonly occurring terms.

Multiple machine learning models were trained and evaluated, including Logistic Regression, Naive Bayes, and Decision Tree. Logistic Regression achieved the best performance due to its ability to handle high-dimensional sparse data efficiently. Naive Bayes provided good results with faster computation, while Decision Tree showed comparatively lower performance.

Overall, this project highlights the importance of a well-structured NLP pipeline. Proper preprocessing, effective feature extraction, and appropriate model selection significantly impact the performance of sentiment analysis systems. This approach can be extended to various real-world applications such as customer feedback analysis, social media monitoring, and product review classification.